# Module 02 : Activations : Intelligence through non-linearity

This module is to add up some ingredient that make neural networks intelligent : **nonlinearity**

In [5]:

import os
print(os.getcwd())


D:\DS\Data Science\TinyTorch\projects\my-tinytorch-systems\notebook


In [71]:
#| default_exp foundation.activations

import numpy as np
rng = np.random.default_rng(7)
from typing import Optional

import sys

# Add project root to path
sys.path.append(r"D:\DS\Data Science\TinyTorch\projects\my-tinytorch-systems")
from tinytorch.foundation.tensor import Tensor

# Constant for numerical comparisons
TOLERANCE = 1e-10  # Small tolerance for floating point comparison in tests

# Export only activation classes
__all__ = ['Sigmoid', 'Relu', 'Tanh', 'GELU', 'Softmax']

In [39]:
# Sigmoid function

class Sigmoid:
    """
    Sigmoid activation: σ(x) = 1/(1 + e^(-x))

    Maps any real number to (0, 1) range.
    Perfect for probabilities and binary classification.
    """
    def parameters() :
        return []

    def forward(self, x: Tensor) -> Tensor:
        x_data = x.data

        # to handle unpredicted
        with np.errstate(over="ignore", invalid="ignore"):
            result = np.where(
                x_data >= 0,
                1.0 / (1.0 + np.exp(-x_data)),
                np.exp(x_data)/ (1 + np.exp(x_data))
            )
        return Tensor(result)    

    def __call__(self, x: Tensor) -> Tensor:
        """Allows the activation to be called like a function."""
        return self.forward(x)
    
    

# 🧪Unit Test Sigmoid

In [41]:
def test_unit_sigmoid():
    """ 🧪 Test Sigmoid implementation """
    print("Unit test : Sigmoid...")
    
    sigmoid = Sigmoid()

    # Test basic case
    x = Tensor([2.0])
    result = sigmoid.forward(x)
    assert np.allclose(result.data, [0.880797]), f"sigmoid(2.0) should be 0.880797, got {result.data}"

    # Test range properly. All outputs should be in (0,1)
    x = Tensor([-10, -1, 0, 1, 10])
    result = sigmoid.forward(x)
    assert np.all(result.data > 0) and np.all(result.data < 1), "All Sigmoid outputs should be in (0,1)"

    # Test extreme values. The stable form must stay finite without overflow
    x = Tensor([-1000, 1000])
    result = sigmoid.forward(x)
    assert np.allclose(result.data[0], 0, atol=TOLERANCE),  "sigmoid(-∞) should approach 0"
    assert np.allclose(result.data[1], 1, atol=TOLERANCE),  "sigmoid(+∞) should approach 1"
    
    print("Test Result : Reussi")

if __name__ == "__main__" :
    test_unit_sigmoid()

Unit test : Sigmoid...
Test Result : Reussi


## ReLU - The sparsity creator

In [42]:
class ReLU:
    """
        ReLU activation : f(x) = max(0,1)
        sets negative values to zero, keeps positive values unchanged
    """

    def parameters(self):
        """ returns empty list : RelU have no learnable parameters """
        return []

    def forward(self, x: Tensor) -> Tensor:
       
        result = np.maximum(0,x.data)
        return Tensor(result)

    def __call__(self, x: Tensor) -> Tensor:
        return self.forward(x)

# Unit test ReLU 

In [59]:
def test_unit_relu():

    """ 🧪 Test ReLU implementation """
    print("Unit test : ReLU...")

    relu = ReLU()

    # Test mixed positive and negative values
    x = Tensor([-2, -1, 0, 1, 2])
    result = relu.forward(x)
    expected = [0, 0, 0, 1, 2]
    #relu(x)
    assert np.allclose(result.data, expected) , f"Relu failed. Expected {expected}; got {relu(x).data}"
    print("Test of Positive and negative values succeed")

    # Test only negative values
    x = Tensor([-2, -1, -5])
    result = relu.forward(x)
    expected = [0, 0, 0]
    assert np.allclose(result.data, expected) , "Relu should zero all negative values"
    print("Test of negative values succeed")

    # Test only negative values
    x = Tensor([2, 1, 5])
    result = relu.forward(x)
    expected = [2, 1, 5]
    assert np.allclose(result.data, expected) , "Relu should preserve all positive values"
    print("Test of positive values succeed")

    # Test sparsity property
    x = Tensor([-1, -2, -3, 1])
    result = relu.forward(x)
    zero = np.sum(result.data == 0)
    assert zero == 3 , f"Relu should create sparsity, got {zero} zeros out of 4"
    print("Test of sparsity values succeed")

    print("\nReLU works correctly")
    
test_unit_relu()

Unit test : ReLU...
Test of Positive and negative values succeed
Test of negative values succeed
Test of positive values succeed
Test of sparsity values succeed

 ReLU works correctly


# Tanh - the zero-centered alternative

In [66]:
class Tanh:
    """ 
        Tanh activation : f(x) = (e^x - e^(-x))/(e^x + e^(-x))

        Maps any real number to (-1,1) range.
        Zero-centered alternatives to sigmoid.
    """

    def parameters(self):
        return []

    def forward(self, x: Tensor) -> Tensor:
        result = np.tanh(x.data) 
        return Tensor(result)

    def __call__(self,  x: Tensor) -> Tensor:
        return self.forward(x)

# Unit test : Tanh

In [74]:
def test_unit_tanh():
   
    """
     🧪 Test tanh implementation
    """
    print("Unit test : Tanh ....")

    tanh = Tanh()
    
    # Test zero
    x = Tensor([0.0])
    expected = [0.0]
    result = tanh(x)
    assert np.allclose(result.data, expected), f"tanh(0) test failed: 0 were expected, we got {result.data}"

    # Test range property
    x = Tensor([-10, -1, 0, 1, 10])
    result = tanh(x)
    assert np.all(result.data >= -1) and np.all(result.data <= 1), f"All tanh should be in [-1, 1]"

    # Test symmetry : tanh(-x) = -tanh(x)
    x = Tensor([2.0])
    pos_result = tanh(x)
    neg_x = Tensor([-2.0])
    neg_result = tanh(neg_x)
    assert np.allclose(pos_result.data, -neg_result.data), "tanh should be symmetric: tanh(-x)=-tanh(x)"

    # Test extreme values
    x = Tensor([-10000, 10000])
    result = tanh(x)
    assert np.allclose(result.data[0], -1, atol=TOLERANCE), "tanh(-∞) should approach -1"
    assert np.allclose(result.data[1], 1, atol=TOLERANCE), "tanh(+∞) should approach 1"

    print("✅ Tanh works correctly!")

if __name__ == "__main__" :
    test_unit_tanh() 

Unit test : Tanh ....
✅ Tanh works correctly!


## GELU - The smooth modern choice

In [75]:
class GELU:
    """
         GELU activation: f(x) = x * Φ(x) ≈ x * Sigmoid(1.702 * x)

        Smooth approximation to ReLU, used in modern architectures.
        Where Φ(x) is the cumulative distribution function of standard normal.
    """

    def parameters(self):
        """Return empty list (activations have no learnable parameters)."""
        return []

    def forward(self, x: Tensor) -> Tensor:
        return Sigmoid()(x * 1.702) * x

    def __call__(self, x: Tensor) -> Tensor:
        return self.forward(x)
        

## Unit test GELU


In [78]:
def test_unit_gelu():

    print("Unit test : GELU ...")
    gelu = GELU()

    # Twest zero(should be approximatively 0)
    x = Tensor([0.0])
    result = gelu(x)
    assert np.allclose(result.data, [0.0], atol=TOLERANCE), f"gelu(0) Should produce 0, got {gelu(x.data)} "

    # Test positive values (should be roughly preserved)
    x = Tensor([1.0])
    assert gelu(x).data[0] > 0.8, f"GELU(1) should be ≈0.84, got {gelu(x).data[0]}"

    # Test negative values (should be small but not zero)
    x = Tensor([-1.0])
    result = gelu.forward(x)
    assert result.data[0] < 0 and result.data[0] > -0.2, f"GELU(-1) should be ≈-0.16, got {result.data[0]}"

    # Test smoothness property (no sharp corners like ReLU)
    x = Tensor([-0.001, 0.0, 0.001])
    result = gelu.forward(x)
    # Values should be close to each other (smooth)
    diff1 = abs(result.data[1] - result.data[0])
    diff2 = abs(result.data[2] - result.data[1])
    assert diff1 < 0.01 and diff2 < 0.01, "GELU should be smooth around zero"

    print("✅ GELU works correctly!")

if __name__ =="__main__":
    test_unit_gelu()

Unit test : GELU ...
✅ GELU works correctly!


## Softmax : The probability distributor

In [79]:
class Softmax:
    """
    Softmax activation: f(x_i) = e^(x_i) / Σ(e^(x_j))

    Converts any vector to a probability distribution.
    Sum of all outputs equals 1.0.
    """

    def parameters(self):
        """Return empty list (activations have no learnable parameters)."""
        return []

    def forward(self, x: Tensor, dim: int = -1) -> Tensor:
        """
        Apply softmax activation along specified dimension.
        """
       
        # Numerical stability: subtract max to prevent overflow
        x_max = np.max(x.data, axis=dim, keepdims=True)
        x_shifted = x.data - x_max

        # Compute exponentials
        exp_values = np.exp(x_shifted)

        # Sum along dimension
        exp_sum = np.sum(exp_values, axis=dim, keepdims=True)

        # Normalize to get probabilities
        result = exp_values / exp_sum
        return Tensor(result)
        

    def __call__(self, x: Tensor, dim: int = -1) -> Tensor:
        """Allows the activation to be called like a function."""
        return self.forward(x, dim)

# Test Unit : Softmax 

In [80]:
def test_unit_softmax():
    """🧪 Test Softmax implementation."""
    print("Unit Test: Softmax...")

    softmax = Softmax()

    # Test basic probability properties
    x = Tensor([1, 2, 3])
    result = softmax.forward(x)

    # Should sum to 1
    assert np.allclose(np.sum(result.data), 1.0), f"Softmax should sum to 1, got {np.sum(result.data)}"

    # All values should be positive
    assert np.all(result.data > 0), "All softmax values should be positive"

    # All values should be less than 1
    assert np.all(result.data < 1), "All softmax values should be less than 1"

    # Largest input should get largest output
    max_input_idx = np.argmax(x.data)
    max_output_idx = np.argmax(result.data)
    assert max_input_idx == max_output_idx, "Largest input should get largest softmax output"

    # Test numerical stability with large numbers
    x = Tensor([1000, 1001, 1002])  # Would overflow without max subtraction
    result = softmax.forward(x)
    assert np.allclose(np.sum(result.data), 1.0), "Softmax should handle large numbers"
    assert not np.any(np.isnan(result.data)), "Softmax should not produce NaN"
    assert not np.any(np.isinf(result.data)), "Softmax should not produce infinity"

    # Test with 2D tensor (batch dimension)
    x = Tensor([[1, 2], [3, 4]])
    result = softmax.forward(x, dim=-1)  # Softmax along last dimension
    assert result.shape == (2, 2), "Softmax should preserve input shape"
    # Each row should sum to 1
    row_sums = np.sum(result.data, axis=-1)
    assert np.allclose(row_sums, [1.0, 1.0]), "Each row should sum to 1"

    print("✅ Softmax works correctly!")

if __name__ == "__main__":
    test_unit_softmax()

Unit Test: Softmax...
✅ Softmax works correctly!


# Integration : Bringint it all together
Lets test how all our activation funtions work together and understand their different behaviors

In [83]:
def test_module():
    """
    Module Test: Complete Integration
    """
    print("RUNNING MODULE INTEGRATION TEST")
    print("=" * 50)

    # Run all unit tests
    print("Running unit tests...")
    test_unit_sigmoid()
    test_unit_relu()
    test_unit_tanh()
    test_unit_gelu()
    test_unit_softmax()

    print("\nRunning integration scenarios...")

    # Test 1: All activations preserve tensor properties
    print("🧪 Integration Test: Tensor property preservation...")
    test_data = Tensor([[1, -1], [2, -2]])  # 2D tensor

    activations = [Sigmoid(), ReLU(), Tanh(), GELU()]
    for activation in activations:
        result = activation.forward(test_data)
        assert result.shape == test_data.shape, f"Shape not preserved by {activation.__class__.__name__}"
        assert isinstance(result, Tensor), f"Output not Tensor from {activation.__class__.__name__}"

    print("✅ All activations preserve tensor properties!")

    # Test 2: Softmax works with different dimensions
    print(" Integration Test: Softmax dimension handling...")
    data_3d = Tensor([[[1, 2, 3], [4, 5, 6]], [[7, 8, 9], [10, 11, 12]]])  # (2, 2, 3)
    softmax = Softmax()

    # Test different dimensions
    result_last = softmax(data_3d, dim=-1)
    assert result_last.shape == (2, 2, 3), "Softmax should preserve shape"

    # Check that last dimension sums to 1
    last_dim_sums = np.sum(result_last.data, axis=-1)
    assert np.allclose(last_dim_sums, 1.0), "Last dimension should sum to 1"

    print("✅ Softmax handles different dimensions correctly!")

    # Test 3: Activation chaining (simulating neural network)
    print("🧪 Integration Test: Activation chaining...")

    # Simulate: Input → Linear → ReLU → Linear → Softmax (like a simple network)
    x = Tensor([[-1, 0, 1, 2]])  # Batch of 1, 4 features

    # Apply ReLU (hidden layer activation)
    relu = ReLU()
    hidden = relu.forward(x)

    # Apply Softmax (output layer activation)
    softmax = Softmax()
    output = softmax.forward(hidden)

    # Verify the chain
    assert hidden.data[0, 0] == 0, "ReLU should zero negative input"
    assert np.allclose(np.sum(output.data), 1.0), "Final output should be probability distribution"

    print("✅ Activation chaining works correctly!")

    print("\n" + "=" * 50)
    print("🎉 ALL TESTS PASSED! Module ready for export.")

# Run module test
if __name__ == "__main__":
    test_module()

RUNNING MODULE INTEGRATION TEST
Running unit tests...
Unit test : Sigmoid...
Test Result : Reussi
Unit test : ReLU...
Test of Positive and negative values succeed
Test of negative values succeed
Test of positive values succeed
Test of sparsity values succeed

 ReLU works correctly
Unit test : Tanh ....
✅ Tanh works correctly!
Unit test : GELU ...
✅ GELU works correctly!
Unit Test: Softmax...
✅ Softmax works correctly!

Running integration scenarios...
🧪 Integration Test: Tensor property preservation...
✅ All activations preserve tensor properties!
 Integration Test: Softmax dimension handling...
✅ Softmax handles different dimensions correctly!
🧪 Integration Test: Activation chaining...
✅ Activation chaining works correctly!

🎉 ALL TESTS PASSED! Module ready for export.


In [84]:
def demo_activations():
    """🎯 See how activations transform data."""
    print("🎯 AHA MOMENT: Activations Add Intelligence")
    print("=" * 45)

    # Test input with positive and negative values
    x = Tensor(np.array([-2.0, -1.0, 0.0, 1.0, 2.0]))
    print(f"Input:   {x.data}")

    # ReLU - zeros out negatives
    relu = ReLU()
    relu_out = relu(x)
    print(f"ReLU:    {relu_out.data}")
    print("         Negatives become 0, positives unchanged!")

    # Sigmoid - squashes to (0, 1)
    sigmoid = Sigmoid()
    sigmoid_out = sigmoid(x)
    print(f"\nSigmoid: {np.round(sigmoid_out.data, 2)}")
    print("         All values squashed to (0, 1) range!")

    # Softmax - probability distribution
    softmax = Softmax()
    softmax_out = softmax(x)
    print(f"\nSoftmax: {np.round(softmax_out.data, 3)}")
    print(f"         Sum = {softmax_out.data.sum():.1f} (valid probability distribution!)")

    print("\n✨ Activations add nonlinearity—the key to deep learning!")

In [85]:
if __name__ == "__main__":
    test_module()
    print("\n")
    demo_activations()

RUNNING MODULE INTEGRATION TEST
Running unit tests...
Unit test : Sigmoid...
Test Result : Reussi
Unit test : ReLU...
Test of Positive and negative values succeed
Test of negative values succeed
Test of positive values succeed
Test of sparsity values succeed

 ReLU works correctly
Unit test : Tanh ....
✅ Tanh works correctly!
Unit test : GELU ...
✅ GELU works correctly!
Unit Test: Softmax...
✅ Softmax works correctly!

Running integration scenarios...
🧪 Integration Test: Tensor property preservation...
✅ All activations preserve tensor properties!
 Integration Test: Softmax dimension handling...
✅ Softmax handles different dimensions correctly!
🧪 Integration Test: Activation chaining...
✅ Activation chaining works correctly!

🎉 ALL TESTS PASSED! Module ready for export.


🎯 AHA MOMENT: Activations Add Intelligence
Input:   [-2. -1.  0.  1.  2.]
ReLU:    [0. 0. 0. 1. 2.]
         Negatives become 0, positives unchanged!

Sigmoid: [0.12 0.27 0.5  0.73 0.88]
         All values squashed t